# Nevus vs. melanoma — SmolVLM + LoRA
In Colab, select **Runtime → Change runtime type → GPU**, then run the sections in order.

We train only LoRA adapters. SmolVLM keeps its original text-generation head and learns to answer **A = nevus** or **B = melanoma**. All original weights stay frozen.

HAM10000 is split by lesion into approximately 70% training, 15% validation, and 15% test. Only `nv` and `mel` images are used. This is a research experiment; lesion grouping does not establish patient independence.

## 1. Install and import
These dependencies are installed in Colab, separately from the local dataset downloader.

In [ ]:
%pip install -q transformers==4.57.1 peft==0.17.1 accelerate==1.11.0 scikit-learn==1.7.2 pandas==2.2.3 pillow==11.3.0 kagglehub==0.3.13 safetensors==0.6.2

In [ ]:
import json
import random
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, roc_auc_score,
)
from transformers import AutoProcessor, Idefics3ForConditionalGeneration
from peft import LoraConfig, PeftModel, get_peft_model, set_peft_model_state_dict
from safetensors.torch import load_file

## 2. Settings
Start with batch size 1 for Colab. Gradient accumulation gives an effective batch size of 16. Use a new run name for each experiment.

In [ ]:
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
RUN_NAME = "smolvlm_nv_mel_lora_seed42"
DATA_DIR = None  # Or the path to an already extracted HAM10000 dataset.
USE_DRIVE = True

SEED = 42
EPOCHS = 5
BATCH_SIZE = 1
ACCUMULATION = 16
LORA_RANK = 8
LEARNING_RATE = 1e-4

assert torch.cuda.is_available(), "Select a GPU runtime in Colab."
DEVICE = torch.device("cuda")
supports_bf16 = torch.cuda.get_device_capability()[0] >= 8
AMP_DTYPE = torch.bfloat16 if supports_bf16 else torch.float16
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(torch.cuda.get_device_name(), AMP_DTYPE)

## 3. Save location
Drive preserves results after Colab disconnects. Existing run folders are rejected to protect earlier results. This notebook saves the best adapters, but does not resume interrupted optimizer state.

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    output_root = Path("/content/drive/MyDrive/comp345")
else:
    output_root = Path("/content/comp345")

OUTPUT_DIR = output_root / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
print(OUTPUT_DIR)

## 4. Load nevus and melanoma images
Download the original JPEGs and metadata. If Kaggle requests authentication, run `kagglehub.login()` and retry. Keep credentials out of notebook cells.

In [ ]:
if DATA_DIR is None:
    DATA_DIR = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
DATA_DIR = Path(DATA_DIR)

frame = pd.read_csv(DATA_DIR / "HAM10000_metadata.csv")
frame = frame[frame.dx.isin(["nv", "mel"])].copy()
frame["label"] = frame.dx.map({"nv": 0, "mel": 1})

# Kaggle includes uppercase/lowercase copies; select one folder per part.
image_folders = []
for part in [1, 2]:
    folder = DATA_DIR / f"HAM10000_images_part_{part}"
    if not folder.is_dir():
        folder = DATA_DIR / f"ham10000_images_part_{part}"
    assert folder.is_dir(), f"Missing image folder: {folder}"
    image_folders.append(folder)
images = [path for folder in image_folders for path in folder.glob("*.jpg")]
image_paths = {p.stem: str(p) for p in images}
assert len(image_paths) == len(images), "Duplicate image IDs in the dataset."
frame["path"] = frame.image_id.map(image_paths)
assert not frame.empty and frame.image_id.is_unique
assert frame[["image_id", "lesion_id", "path", "label"]].notna().all().all()
assert frame.groupby("lesion_id").label.nunique().max() == 1
print(frame.dx.value_counts())

## 5. Split by lesion
Every view of a lesion stays in one split. We stratify lesion labels; image proportions may differ slightly from 70/15/15. Saved manifests keep later artifact experiments on the same split.

In [ ]:
lesions = frame[["lesion_id", "label"]].drop_duplicates().sort_values("lesion_id")
train_lesions, heldout = train_test_split(
    lesions, test_size=0.30, stratify=lesions.label, random_state=SEED,
)
val_lesions, test_lesions = train_test_split(
    heldout, test_size=0.50, stratify=heldout.label, random_state=SEED,
)

splits = {}
for name, groups in [("train", train_lesions), ("validation", val_lesions), ("test", test_lesions)]:
    splits[name] = frame[frame.lesion_id.isin(groups.lesion_id)].reset_index(drop=True)
    assert set(splits[name].label) == {0, 1}
    splits[name].to_csv(OUTPUT_DIR / f"{name}.csv", index=False)

for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    assert set(splits[a].lesion_id).isdisjoint(splits[b].lesion_id)
assert sum(map(len, splits.values())) == len(frame)
display(pd.DataFrame({name: data.dx.value_counts() for name, data in splits.items()}))

## 6. Image processor and answer tokens
Each image is resized to one 512×512 square, with no crops, augmentation, or artifact removal. Every image gets the same prompt; the ground-truth label is never included in the prompt. Single-token answers avoid multi-token label scoring.

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"
processor.image_processor.do_image_splitting = False
processor.image_processor.size = {"longest_edge": 512}
processor.image_processor.max_image_size = {"longest_edge": 512}

PROMPT = processor.apply_chat_template([
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": "Classify this skin lesion. A: nevus. B: melanoma. Answer only A or B."},
    ]},
], add_generation_prompt=True)

answer_ids = [processor.tokenizer.encode(a, add_special_tokens=False) for a in ["A", "B"]]
assert all(len(ids) == 1 for ids in answer_ids), "Answers must be single tokens."
assert answer_ids[0] != answer_ids[1]
ANSWER_TOKEN_IDS = torch.tensor([ids[0] for ids in answer_ids], device=DEVICE)
processor.save_pretrained(OUTPUT_DIR / "processor")

## 7. Data loaders
The dataset opens images as needed. The batch function prepares image and prompt tensors together; it does not truncate image tokens.

In [ ]:
class LesionDataset(Dataset):
    def __init__(self, table):
        self.table = table

    def __len__(self):
        return len(self.table)

    def __getitem__(self, index):
        row = self.table.iloc[index]
        with Image.open(row.path) as image:
            return image.convert("RGB"), int(row.label)


def collate(examples):
    images, labels = zip(*examples)
    inputs = processor(
        text=[PROMPT] * len(images), images=[[image] for image in images],
        padding=True, return_tensors="pt",
    )
    return inputs, torch.tensor(labels)


loaders = {
    name: DataLoader(LesionDataset(data), batch_size=BATCH_SIZE,
                     shuffle=(name == "train"), collate_fn=collate)
    for name, data in splits.items()
}

## 8. Load SmolVLM and add LoRA
Only the language model's query and value adapters train. The vision encoder and original vocabulary head stay frozen. FP32 weights with mixed-precision computation support FP16 training on a T4.

In [ ]:
model = Idefics3ForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float32, attn_implementation="sdpa",
)
model.requires_grad_(False)
model.config.use_cache = False
model.model.text_model.config.use_cache = False

targets = [
    name for name, layer in model.named_modules()
    if name.startswith("model.text_model.")
    and name.endswith(("q_proj", "v_proj")) and isinstance(layer, nn.Linear)
]
assert targets
model = get_peft_model(model, LoraConfig(
    r=LORA_RANK, lora_alpha=2 * LORA_RANK, lora_dropout=0.05,
    target_modules=targets, bias="none",
))
model.get_base_model().model.text_model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False},
)
model = model.to(DEVICE)
assert all("lora_" in name for name, p in model.named_parameters() if p.requires_grad)
model.print_trainable_parameters()

## 9. Prediction and evaluation helpers
We use the next-token vocabulary logits. Training learns the correct answer across the full vocabulary; evaluation compares only `A` and `B`, equivalent to constrained one-token generation. The resulting melanoma score is conditional on those choices and is not a calibrated disease probability. Metrics use a fixed threshold of 0.5.

In [ ]:
def next_token_logits(model, inputs):
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
    # Left padding puts each prompt's final token at the last position.
    position_ids = inputs["attention_mask"].long().cumsum(-1) - 1
    position_ids.clamp_(min=0)
    return model(**inputs, position_ids=position_ids, use_cache=False,
                 logits_to_keep=1).logits[:, -1].float()

In [ ]:
@torch.inference_mode()
def evaluate(loader):
    model.eval()
    labels, scores = [], []
    for inputs, y in tqdm(loader, desc="Evaluate", leave=False):
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE):
            logits = next_token_logits(model, inputs)
        scores.extend(logits[:, ANSWER_TOKEN_IDS].softmax(-1)[:, 1].cpu().tolist())
        labels.extend(y.tolist())

    predictions = (np.array(scores) >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    metrics = {
        "accuracy": accuracy_score(labels, predictions),
        "balanced_accuracy": balanced_accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0),
        "melanoma_recall": float(tp / max(tp + fn, 1)),
        "melanoma_precision": float(tp / max(tp + fp, 1)),
        "nevus_specificity": float(tn / max(tn + fp, 1)),
        "roc_auc": roc_auc_score(labels, scores),
        "pr_auc_average_precision": average_precision_score(labels, scores),
        "confusion_matrix_nv_mel": [[int(tn), int(fp)], [int(fn), int(tp)]],
    }
    return metrics, scores

## 10. Training helpers
Class weights are calculated from training data only and applied per example, so they still work with batch size 1. Gradients are accumulated over 16 batches, including a correctly scaled final partial window.

In [ ]:
counts = np.bincount(splits["train"].label, minlength=2)
class_weights = torch.tensor(counts.sum() / (2 * counts), device=DEVICE, dtype=torch.float32)
trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=LEARNING_RATE, weight_decay=0.01)
scaler = torch.amp.GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))


def train_one_epoch(loader):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    total_loss = 0
    for step, (inputs, labels) in enumerate(tqdm(loader, desc="Train")):
        labels = labels.to(DEVICE)
        window_start = (step // ACCUMULATION) * ACCUMULATION
        window_examples = min(ACCUMULATION * BATCH_SIZE,
                              len(loader.dataset) - window_start * BATCH_SIZE)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE):
            logits = next_token_logits(model, inputs)
            losses = F.cross_entropy(logits, ANSWER_TOKEN_IDS[labels], reduction="none")
            losses = losses * class_weights[labels]
        scaler.scale(losses.sum() / window_examples).backward()
        total_loss += losses.detach().sum().item()

        if (step + 1) % ACCUMULATION == 0 or step + 1 == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
    return total_loss / len(loader.dataset)

## 11. Train and save the best adapters
Validation PR-AUC (average precision) selects the checkpoint. The test set is left untouched until the next section. Class weighting can affect score calibration.

In [ ]:
config = {
    "model_id": MODEL_ID, "seed": SEED, "epochs": EPOCHS,
    "batch_size": BATCH_SIZE, "accumulation": ACCUMULATION,
    "lora_rank": LORA_RANK, "learning_rate": LEARNING_RATE,
    "prompt": PROMPT, "answer_token_ids": ANSWER_TOKEN_IDS.tolist(),
    "label2id": {"nv": 0, "mel": 1}, "class_weights": class_weights.tolist(),
}
(OUTPUT_DIR / "config.json").write_text(json.dumps(config, indent=2))

best_score = -1
history = []
for epoch in range(EPOCHS):
    train_loss = train_one_epoch(loaders["train"])
    validation, _ = evaluate(loaders["validation"])
    record = {"epoch": epoch + 1, "train_loss": train_loss, **validation}
    history.append(record)
    print(json.dumps(record, indent=2))
    (OUTPUT_DIR / "history.json").write_text(json.dumps(history, indent=2))

    if validation["pr_auc_average_precision"] > best_score:
        best_score = validation["pr_auc_average_precision"]
        model.save_pretrained(OUTPUT_DIR / "best_adapter")
        (OUTPUT_DIR / "best_validation.json").write_text(json.dumps(record, indent=2))

## 12. Test the selected model
Evaluate once after training. Metrics are per image, and views of the same lesion are correlated. Compare PR-AUC with melanoma prevalence. Keep this split fixed for artifact experiments; do not tune on test results.

In [ ]:
adapter_state = load_file(str(OUTPUT_DIR / "best_adapter" / "adapter_model.safetensors"))
set_peft_model_state_dict(model, adapter_state)

test_metrics, test_scores = evaluate(loaders["test"])
test_metrics["melanoma_prevalence"] = float(splits["test"].label.mean())
(OUTPUT_DIR / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2))

predictions = splits["test"][["image_id", "lesion_id", "dx", "label"]].copy()
predictions["conditional_melanoma_score"] = test_scores
predictions["prediction"] = (np.array(test_scores) >= 0.5).astype(int)
predictions.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)
print(json.dumps(test_metrics, indent=2))

## 13. Predict a single image
Uses the selected adapters loaded above. The score compares `A` and `B`; it is not a calibrated medical probability.

In [ ]:
@torch.inference_mode()
def predict_image(path):
    model.eval()
    with Image.open(path) as image:
        inputs = processor(text=PROMPT, images=[image.convert("RGB")], return_tensors="pt")
    with torch.autocast(DEVICE.type, dtype=AMP_DTYPE):
        logits = next_token_logits(model, inputs)
    score = logits[:, ANSWER_TOKEN_IDS].softmax(-1)[0, 1].item()
    return {"prediction": "mel" if score >= 0.5 else "nv",
            "conditional_melanoma_score": score}


# predict_image("/content/example.jpg")

## 14. Optional: reload a saved run
In a fresh GPU runtime, run sections **1–2**, this cell, the **first code cell in section 9**, and section **13**. You can then call `predict_image(path)` without downloading the dataset or creating a new run folder.


In [ ]:
# Uncomment to reload a completed run. Mount Drive first if needed.
# from google.colab import drive
# drive.mount("/content/drive")
# SAVED_RUN = Path("/content/drive/MyDrive/comp345/smolvlm_nv_mel_lora_seed42")
# config = json.loads((SAVED_RUN / "config.json").read_text())
# processor = AutoProcessor.from_pretrained(SAVED_RUN / "processor")
# PROMPT = config["prompt"]
# ANSWER_TOKEN_IDS = torch.tensor(config["answer_token_ids"], device=DEVICE)
# base_model = Idefics3ForConditionalGeneration.from_pretrained(
#     config["model_id"], torch_dtype=torch.float32, attn_implementation="sdpa",
# )
# model = PeftModel.from_pretrained(base_model, SAVED_RUN / "best_adapter").to(DEVICE)
# del base_model

References: [SmolVLM](https://huggingface.co/HuggingFaceTB/SmolVLM-500M-Instruct) · [PEFT](https://huggingface.co/docs/peft/developer_guides/custom_models) · [HAM10000](https://pmc.ncbi.nlm.nih.gov/articles/PMC6091241/) · [Kaggle dataset](https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000).